In [4]:
import sys
import os
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

sys.path.append(os.path.abspath('..'))
# All metrics now come from the lean utils.py — no SPARQL, no cache
from utils import (
    avg_prf_at_k,
    avg_mrr_at_k,
    avg_hits_at_k,
    avg_jaccard_at_k,   # ← added
    avg_ndcg_at_k,
)

In [5]:
file_path = "../dataset/hard_triplets_tfidf_dbpedia.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

unique_queries = set()
eval_items     = []   # list of (anchor, pool, positives_set)

for item in data:
    anchor    = item["anchor_query"]
    positives = item["positive_queries"]   # already selected by Jaccard >= threshold at build time
    negatives = item["negative_queries"]

    pool = positives + negatives

    unique_queries.add(anchor)
    unique_queries.update(positives)
    unique_queries.update(negatives)

    # positives cast to frozenset — membership test is the relevance label
    eval_items.append((anchor, pool, set(positives)))

queries_list = list(unique_queries)
print(f"Loaded {len(eval_items)} evaluation items.")
print(f"Unique queries in TF-IDF vocabulary: {len(queries_list)}")

Loaded 337 evaluation items.
Unique queries in TF-IDF vocabulary: 1677


In [6]:
# Binary relevance: 1.0 if candidate is in positive_queries, 0.0 otherwise.
# No SPARQL endpoint, no cache, no safe_jaccard_for_queries needed.
# Membership in positive_queries already encodes Jaccard >= threshold (decided at build time).

relevance_scores_per_item = []

for anchor, pool, positives in eval_items:
    scores = {cand: (1.0 if cand in positives else 0.0) for cand in pool}
    relevance_scores_per_item.append(scores)

print(f"Binary relevance scores built for {len(relevance_scores_per_item)} items. "
      f"No SPARQL calls needed.")

Binary relevance scores built for 337 items. No SPARQL calls needed.


In [7]:
def build_tfidf_rankings(queries, eval_items):
    """
    queries    : list of all unique query strings (vocabulary).
    eval_items : list of (anchor, pool, positives).
    Returns    : list of ranked candidate lists, one per eval item.
    """
    vectorizer = TfidfVectorizer(
        analyzer      = "word",
        ngram_range   = (1, 2),   # unigrams + bigrams
        sublinear_tf  = True,     # log(1+tf) dampening
        min_df        = 2,        # drop singleton tokens
        max_df        = 0.95,     # drop near-universal tokens (SELECT, WHERE, {})
        norm          = "l2",     # unit vectors → cosine = dot product
        smooth_idf    = True,
        token_pattern = r"(?u)\S+",  # keeps wdt:P31 and wd:Q5 intact
    )

    tfidf_matrix = normalize(vectorizer.fit_transform(queries))
    query_to_row = {q: i for i, q in enumerate(queries)}
    ranked_pools = []

    for anchor, pool, _ in eval_items:
        anchor_idx = query_to_row.get(anchor)
        if anchor_idx is None:
            ranked_pools.append([])
            continue

        cand_indices, cand_queries = [], []
        for q in pool:
            idx = query_to_row.get(q)
            if idx is not None:
                cand_indices.append(idx)
                cand_queries.append(q)

        if not cand_indices:
            ranked_pools.append([])
            continue

        anchor_vec = tfidf_matrix[anchor_idx]
        cand_vecs  = tfidf_matrix[cand_indices]
        sims       = (cand_vecs @ anchor_vec.T).toarray().ravel()

        sorted_candidates = [cand_queries[i] for i in np.argsort(-sims)]
        ranked_pools.append(sorted_candidates)

    return ranked_pools

In [8]:
print("Building TF-IDF rankings...")
tfidf_sorted_pools = build_tfidf_rankings(queries_list, eval_items)
print("Done.")

Building TF-IDF rankings...
Done.


In [9]:
TOPKS = [3, 5, 10, 20]

print("=" * 65)
print("FINAL EVALUATION RESULTS (TF-IDF baseline)")
print("=" * 65)

tfidf_sorted_pools = build_tfidf_rankings(queries_list, eval_items)

print("\n[Binary relevance: Annotated Positives]")
print(f"{'':5} {'P':>7} {'R':>7} {'F1':>7} {'MRR':>7} {'Hits':>7} {'avgJacc':>9} {'nDCG':>7}")
print("-" * 62)
for k in TOPKS:
    p, r, f1  = avg_prf_at_k(tfidf_sorted_pools, eval_items, k)
    mrr        = avg_mrr_at_k(tfidf_sorted_pools, eval_items, k)
    hits       = avg_hits_at_k(tfidf_sorted_pools, eval_items, k)
    avg_jacc   = avg_jaccard_at_k(tfidf_sorted_pools, k, relevance_scores_per_item)
    ndcg       = avg_ndcg_at_k(tfidf_sorted_pools, k, relevance_scores_per_item)
    print(f"@{k:<4} {p:>7.4f} {r:>7.4f} {f1:>7.4f} {mrr:>7.4f} {hits:>7.4f} {avg_jacc:>9.4f} {ndcg:>7.4f}")

FINAL EVALUATION RESULTS (TF-IDF baseline)

[Binary relevance: Annotated Positives]
            P       R      F1     MRR    Hits   avgJacc    nDCG
--------------------------------------------------------------
@3     0.4313  0.1784  0.2295  0.4461  0.4866    0.4313  0.4286
@5     0.4036  0.2509  0.2793  0.4557  0.5282    0.4036  0.4304
@10    0.3439  0.3606  0.3183  0.4582  0.5460    0.3439  0.4352
@20    0.2961  0.6327  0.3611  0.4786  0.8902    0.2961  0.5221
